# Sampling-based planning with a factor graph

A control proposal determines which trajectories the planner explores. An update rule determines how the next proposal uses the graph errors of those trajectories. This guide makes the two choices independently on a one-dimensional double integrator. The same calls work on a manifold by supplying velocities in its local tangent coordinates.

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia. See LICENSE. Authors: Frank Dellaert and Akash Ratheesh.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/planning/doc/SamplingSearch.ipynb)

In [1]:
import sys
if 'google.colab' in sys.modules:
    %pip install --quiet gtsam-develop plotly

In [2]:
import numpy as np
import plotly.graph_objects as go
import gtsam
from gtsam.symbol_shorthand import U, X
from IPython.display import HTML, display

## Specify dynamics and the objective

The state is position and velocity, and the action is acceleration. A terminal factor asks the system to move one meter and come to rest. The other factors penalize acceleration. Every candidate has a complete set of graph values, so a cost factor may also connect nonconsecutive states.

In [3]:
horizon, dt = 12, 0.15
states = [X(k) for k in range(horizon + 1)]
actions = [U(k) for k in range(horizon)]
planner = gtsam.SamplingPlannerVectorVector(states, actions, dt)
initial, graph = gtsam.Values(), gtsam.NonlinearFactorGraph()
initial.insert(states[0], np.zeros(2))
effort = gtsam.noiseModel.Isotropic.Sigma(1, 6.0)
for key in actions:
    initial.insert(key, np.zeros(1))
    graph.add(gtsam.PriorFactorVector(key, np.zeros(1), effort))
goal = np.array([1.0, 0.0])
terminal = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.08, 0.15]))
graph.add(gtsam.PriorFactorVector(states[-1], goal, terminal))


def dynamics(state, acceleration):
    return np.array([state[1], acceleration[0]])

## Inspect one optimization step at a time

Here CEM refits the mean and marginal standard deviations of a Gaussian proposal from eight elites. Keeping the search state lets us inspect the cost of its nominal trajectory after every update.

In [4]:
proposal = gtsam.GaussianProposal(np.array([[2.0]]), 64, 17)
rule = gtsam.CEM(8)
search = proposal.initialize(planner.nominalControls(initial))


def evaluate(controls):
    return graph.error(planner.rollout(initial, controls, dynamics))


history = [evaluate(search.mean)]
for iteration in range(5):
    planner.iterate(graph, initial, dynamics, proposal, rule, search)
    history.append(evaluate(search.mean))
print(f"Initial cost {history[0]:.2f}; final cost {history[-1]:.2f}")
assert search.iteration == 5 and history[-1] < history[0]

Initial cost 78.12; final cost 0.44


## Change the update rule

MPPI weights each sequence by an exponential function of its cost. Predictive sampling takes the best sampled sequence and includes the incumbent in the competition. Both use the same Gaussian proposal as CEM. The result of `optimize` is a `Values` containing the states and controls of the final plan.

In [5]:
for name, algorithm in [
    ("MPPI", gtsam.MPPI()),
    ("Predictive sampling", gtsam.PredictiveSampling()),
]:
    sampler = gtsam.GaussianProposal(np.array([[2.0]]), 64, 17)
    result = planner.optimize(graph, initial, dynamics, sampler, algorithm, 5)
    print(name, "final [position, velocity]", result.atVector(states[-1]).round(3))
    assert graph.error(result) < history[0]

MPPI final [position, velocity] [1.012 0.02 ]


Predictive sampling final [position, velocity] [1.02  0.034]


The plot follows the cost of the CEM mean after each update. An individual iteration can increase this cost because CEM fits a distribution to several elites. Interactive charts require an internet connection.

In [6]:
figure = go.Figure(
    go.Scatter(x=list(range(len(history))), y=history, mode="lines+markers")
)
figure.update_layout(
    xaxis_title="CEM iteration", yaxis_title="Nominal trajectory graph error"
)
from html import escape

document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Sampling-based planning with a factor graph" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



For the C++ signatures, proposal families and weighting conventions, see the [architecture overview](PlanningOverview.md).